# dbt Deep — Materializations, Incremental, Snapshots, Tests

## Mental Model

dbt is a SQL transformation compiler and execution framework. In production, the most important question is not only **what** a model computes, but **how** it materializes and **how much data** it has to reprocess.

This notebook demonstrates four production-grade dbt ideas against the **Citi dbt project (`citi_dbt`)**:

1. **Materializations** — `table`, `view`, `ephemeral`, `incremental`
2. **Incremental models** — process only new deltas instead of full history
3. **Snapshots** — preserve slowly changing state over time
4. **Tests + lineage** — validate data contracts and show dependency structure

### Citi context
- 6,000+ API endpoints monitored for latency, error rate, and throughput
- Alerts escalate through severity tiers
- PostgreSQL powers the local learning stack
- dbt project name: `citi_dbt`


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import textwrap
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor

DBT_PATH = r"C:/py_venv/proj_educate/Scripts/dbt.exe"
PROJECT_DIR = Path.home() / "citi_dbt"
MODELS_DIR = PROJECT_DIR / "models"
SNAPSHOTS_DIR = PROJECT_DIR / "snapshots"
TARGET_DIR = PROJECT_DIR / "target"

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

TARGET_NAME = "postgres"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 200)

print(f"DBT_PATH      : {DBT_PATH}")
print(f"PROJECT_DIR   : {PROJECT_DIR}")
print(f"MODELS_DIR    : {MODELS_DIR}")
print(f"SNAPSHOTS_DIR : {SNAPSHOTS_DIR}")
print(f"TARGET_DIR    : {TARGET_DIR}")


In [ ]:
def ensure_directories() -> None:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    SNAPSHOTS_DIR.mkdir(parents=True, exist_ok=True)
    TARGET_DIR.mkdir(parents=True, exist_ok=True)

def write_text_file(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")

def run_cmd(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    print("RUN >", " ".join(shlex.quote(str(x)) for x in cmd))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(cmd)}")
    return completed

def run_dbt(args: List[str], check: bool = True) -> subprocess.CompletedProcess:
    cmd = [DBT_PATH, *args, "--project-dir", str(PROJECT_DIR), "--target", TARGET_NAME]
    return run_cmd(cmd, cwd=PROJECT_DIR, check=check)

def pg_conn():
    return psycopg2.connect(**DB_CONFIG)

def fetch_df(sql: str, params: Optional[tuple] = None) -> pd.DataFrame:
    with pg_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

def execute_sql(sql: str, params: Optional[tuple] = None) -> None:
    with pg_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
        conn.commit()

def latest_run_results() -> Dict[str, Any]:
    path = TARGET_DIR / "run_results.json"
    if not path.exists():
        raise FileNotFoundError(f"run_results.json not found at {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def flatten_dbt_results(run_results: Dict[str, Any]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for item in run_results.get("results", []):
        node = item.get("node", {})
        rows.append(
            {
                "unique_id": node.get("unique_id"),
                "name": node.get("name"),
                "resource_type": node.get("resource_type"),
                "materialized": (node.get("config") or {}).get("materialized"),
                "status": item.get("status"),
                "execution_time_seconds": item.get("execution_time"),
                "message": item.get("message"),
                "adapter_response": json.dumps(item.get("adapter_response", {})),
            }
        )
    return pd.DataFrame(rows)

ensure_directories()


## 1) Create dbt models for materialization comparison

We generate four models under `citi_dbt/models/`:

- `table_model.sql`
- `view_model.sql`
- `ephemeral_model.sql`
- `incremental_model.sql`

Each is based on `public.alerts`, but materialized differently so we can compare runtime behavior and downstream effects.


In [ ]:
table_model_sql = """
{{ config(materialized='table') }}

select
    alert_id,
    endpoint_id,
    severity,
    message,
    created_at
from public.alerts
"""

view_model_sql = """
{{ config(materialized='view') }}

select
    alert_id,
    endpoint_id,
    severity,
    message,
    created_at
from public.alerts
"""

ephemeral_model_sql = """
{{ config(materialized='ephemeral') }}

select
    alert_id,
    endpoint_id,
    severity,
    message,
    created_at
from public.alerts
"""

incremental_model_sql = """
{{ config(
    materialized='incremental',
    unique_key='alert_id',
    on_schema_change='sync_all_columns'
) }}

select
    alert_id,
    endpoint_id,
    severity,
    message,
    created_at
from public.alerts
{% if is_incremental() %}
where created_at > (
    select coalesce(max(created_at), cast('1900-01-01' as timestamp))
    from {{ this }}
)
{% endif %}
"""

ephemeral_consumer_sql = """
{{ config(materialized='view') }}

with e as (
    select * from {{ ref('ephemeral_model') }}
)
select
    alert_id,
    endpoint_id,
    severity,
    message,
    created_at
from e
"""

schema_yml = """
version: 2

models:
  - name: table_model
    columns:
      - name: alert_id
        tests:
          - not_null
          - unique
      - name: endpoint_id
        tests:
          - not_null
          - relationships:
              to: source('public', 'endpoints')
              field: endpoint_id
      - name: severity
        tests:
          - not_null
          - accepted_values:
              values: ['HIGH', 'CRITICAL', 'MEDIUM', 'LOW']

  - name: incremental_model
    columns:
      - name: alert_id
        tests:
          - not_null
          - unique
      - name: endpoint_id
        tests:
          - not_null
          - relationships:
              to: source('public', 'endpoints')
              field: endpoint_id
      - name: severity
        tests:
          - not_null
          - accepted_values:
              values: ['HIGH', 'CRITICAL', 'MEDIUM', 'LOW']

sources:
  - name: public
    schema: public
    tables:
      - name: endpoints
      - name: alerts
"""

write_text_file(MODELS_DIR / "table_model.sql", table_model_sql)
write_text_file(MODELS_DIR / "view_model.sql", view_model_sql)
write_text_file(MODELS_DIR / "ephemeral_model.sql", ephemeral_model_sql)
write_text_file(MODELS_DIR / "ephemeral_consumer.sql", ephemeral_consumer_sql)
write_text_file(MODELS_DIR / "incremental_model.sql", incremental_model_sql)
write_text_file(MODELS_DIR / "schema.yml", schema_yml)

print("Created / updated model files:")
for p in sorted(MODELS_DIR.glob("*")):
    print(" -", p.name)


## 2) Run dbt models and compare materialization behavior


In [ ]:
run_dbt(["deps"], check=False)
run_dbt(["debug"])

materialization_run = run_dbt(
    [
        "run",
        "--select",
        "table_model view_model ephemeral_consumer incremental_model",
    ]
)

run_results = latest_run_results()
materialization_df = flatten_dbt_results(run_results)

display(
    materialization_df[
        ["name", "resource_type", "materialized", "status", "execution_time_seconds", "message"]
    ].sort_values(["resource_type", "name"]).reset_index(drop=True)
)


### Quick interpretation

- **table**: persists physical data
- **view**: stores logic only
- **ephemeral**: compiles inline into downstream SQL and does not create a standalone relation
- **incremental**: persists data and can process only new rows on later runs


In [ ]:
artifact_check_sql = """
select table_schema, table_name, table_type
from information_schema.tables
where table_schema not in ('pg_catalog', 'information_schema')
  and table_name in ('table_model', 'view_model', 'ephemeral_model', 'ephemeral_consumer', 'incremental_model')
order by table_name
"""
artifacts_df = fetch_df(artifact_check_sql)
display(artifacts_df)

print("Notice: ephemeral_model should not exist as a standalone database object.")


## 3) Incremental model deep dive

We run the incremental model twice:

1. First run builds the full target table
2. We insert one synthetic new alert into `public.alerts`
3. Second run should process only the new row

That is the core production win: **delta-only work**.


In [ ]:
run_dbt(["run", "--full-refresh", "--select", "incremental_model"])

first_count_df = fetch_df("select count(*) as row_count from public.incremental_model")
first_count = int(first_count_df.iloc[0]["row_count"])
print(f"First run row count: {first_count}")

max_id_df = fetch_df("select coalesce(max(alert_id), 0) as max_id from public.alerts")
max_alert_id = int(max_id_df.iloc[0]["max_id"])
new_alert_id = max_alert_id + 1

new_alert_insert_sql = """
insert into public.alerts (alert_id, endpoint_id, severity, message, created_at)
select
    %s as alert_id,
    e.endpoint_id,
    'HIGH' as severity,
    'Synthetic incremental test alert generated by notebook' as message,
    now() + interval '1 minute' as created_at
from public.endpoints e
order by e.endpoint_id
limit 1
"""
execute_sql(new_alert_insert_sql, (new_alert_id,))
print(f"Inserted synthetic alert_id={new_alert_id} into public.alerts")

run_dbt(["run", "--select", "incremental_model"])

second_count_df = fetch_df("select count(*) as row_count from public.incremental_model")
second_count = int(second_count_df.iloc[0]["row_count"])
delta = second_count - first_count

print(f"First run: {first_count} rows, Second run: {second_count} rows (incremental)")
print(f"Delta processed in target table: {delta}")


In [ ]:
latest_incremental_rows = fetch_df(
    """
    select *
    from public.incremental_model
    order by created_at desc, alert_id desc
    limit 5
    """
)
display(latest_incremental_rows)


## 4) Snapshot for endpoint status tracking

We create a snapshot on `public.endpoints` using:

- `strategy='check'`
- `check_cols=['status']`
- `unique_key='endpoint_id'`

Then we update one endpoint's status, rerun the snapshot, and inspect the `dbt_scd_*` tracking columns.


In [ ]:
snapshot_sql = """
{% snapshot endpoints_status_snapshot %}

{{
    config(
      target_schema='snapshots',
      unique_key='endpoint_id',
      strategy='check',
      check_cols=['status']
    )
}}

select
    endpoint_id,
    name,
    region,
    status,
    category
from public.endpoints

{% endsnapshot %}
"""

write_text_file(SNAPSHOTS_DIR / "endpoints_status_snapshot.sql", snapshot_sql)
print("Snapshot file written:", (SNAPSHOTS_DIR / "endpoints_status_snapshot.sql").name)


In [ ]:
run_dbt(["snapshot", "--select", "endpoints_status_snapshot"])

before_change_df = fetch_df(
    """
    select endpoint_id, status
    from public.endpoints
    order by endpoint_id
    limit 1
    """
)
endpoint_id_to_change = int(before_change_df.iloc[0]["endpoint_id"])
old_status = str(before_change_df.iloc[0]["status"])

status_cycle = {
    "ACTIVE": "DEGRADED",
    "DEGRADED": "MAINTENANCE",
    "MAINTENANCE": "ACTIVE",
    "DOWN": "ACTIVE",
}
new_status = status_cycle.get(old_status.upper(), "DEGRADED")

execute_sql(
    "update public.endpoints set status = %s where endpoint_id = %s",
    (new_status, endpoint_id_to_change),
)

print(f"Updated endpoint_id={endpoint_id_to_change} from status={old_status} to status={new_status}")

run_dbt(["snapshot", "--select", "endpoints_status_snapshot"])

snapshot_df = fetch_df(
    """
    select *
    from snapshots.endpoints_status_snapshot
    where endpoint_id = %s
    order by dbt_valid_from
    """,
    (endpoint_id_to_change,),
)
display(snapshot_df)


## 5) Add and run dbt tests

We validate:

- `not_null`
- `unique`
- `accepted_values`
- `relationships`

This is the production data contract layer.


In [ ]:
test_run = run_dbt(["test", "--select", "table_model incremental_model"])

test_results = latest_run_results()
test_df = flatten_dbt_results(test_results)

if test_df.empty:
    print("No test results parsed from run_results.json")
else:
    parsed = test_df[["unique_id", "name", "status", "execution_time_seconds", "message"]].copy()
    parsed["pass_fail"] = parsed["status"].apply(lambda x: "PASS" if str(x).lower() == "pass" else "FAIL")
    display(parsed.reset_index(drop=True))


## 6) Show lineage with `dbt ls`

`dbt ls --select alerts+` shows resources downstream from `alerts`.  
In a dbt DAG:

- nodes = models, tests, snapshots, sources, exposures, seeds
- edges = `ref()` and `source()` dependencies

Lineage matters because production impact analysis depends on it: when a source changes, you can see what downstream models and tests are affected.


In [ ]:
lineage_cmd = run_dbt(["ls", "--select", "source:public.alerts+"])
lineage_lines = [line.strip() for line in lineage_cmd.stdout.splitlines() if line.strip()]
lineage_df = pd.DataFrame({"dbt_resource": lineage_lines})
display(lineage_df)

print("\nSimple DAG interpretation:")
for item in lineage_lines:
    print(" -", item)


## 7) What just happened

**Incremental models are the most important dbt concept for production.**  
They turn a full-refresh job into a delta-only job.

In this Citi scenario, dbt models over the alerts domain can avoid rescanning full history on every run and instead process only newly arrived alerts since the previous successful execution.

That is how batch SQL workflows start behaving more like production-grade pipelines:
- less I/O
- lower runtime
- lower warehouse/database load
- easier scaling
- safer repeated execution


In [ ]:
summary = {
    "project": "citi_dbt",
    "database": "de_telemetry",
    "materializations_demoed": ["table", "view", "ephemeral", "incremental"],
    "snapshot": "endpoints_status_snapshot",
    "tests_added": ["not_null", "unique", "accepted_values", "relationships"],
    "key_message": "Incremental models reduce full-history reprocessing by applying delta-only logic.",
}
summary
